In [1]:
import pandas as pd
from google.cloud import bigquery

In [2]:
client = bigquery.Client(project="taskflow-growth-analytics")

In [4]:
query = """
SELECT *
FROM `taskflow-growth-analytics.taskflow_dev.exp_onboarding_assignment`
"""
df_onboarding = client.query(query).to_dataframe()

In [5]:
df_onboarding.head()

,account_id,variant,did_activate
0,604,control,False
1,668,control,False
2,1046,control,False
3,1147,control,False
4,1732,control,False


In [6]:
summary = df_onboarding.groupby("variant")["did_activate"].agg(["mean", "count", "sum"])
summary

,mean,count,sum
variant,,,
control,0.330214,748,247
treatment,0.329517,786,259


In [7]:
from statsmodels.stats.proportion import proportions_ztest

# counts of activated accounts in each section
activated_counts = [247, 259]

# total accounts in each section
total_counts = [748, 786]

z_stat, p_value = proportions_ztest(count=activated_counts, nobs=total_counts)

print(f"Z-statistic: {z_stat:.4f}, p-value: {p_value:.4f}")

Z-statistic: 0.0290, p-value: 0.9768


In [8]:
from statsmodels.stats.proportion import confint_proportions_2indep

# difference in activation rates (treatment - control), with 95% CI
ci_low, ci_upp = confint_proportions_2indep(
    count1=259, nobs1=786, # treatment: activated, total
    count2=247, nobs2=748, # control: activated, total
    alpha=0.05, method="wald"
)

print(f"95% CI for (treatment - control): ({ci_low:.4f}, {ci_upp:.4f})")

95% CI for (treatment - control): (-0.0478, 0.0464)


In [9]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# effect size for going from 32% to 36% activation
effect = proportion_effectsize(0.36, 0.32)

analysis = NormalIndPower()
required_n = analysis.solve_power(
    effect_size=effect,
    alpha=0.05,
    power=0.80,
    alternative="two-sided"
)

print(f"Required sample size per section: {required_n:.0f}")

Required sample size per section: 2200


## Experiment 1: Onboarding Redesign — Results

**Question:** Did the 3-step onboarding redesign improve activation vs the 5-step control?

**Result:** Control 33.0% vs Treatment 33.0% activation. Two-proportion z-test: p = 0.977.

**Confidence interval** (treatment − control): [−4.8pp, +4.6pp] — very wide, contains 0.

**Conclusion:** No evidence of an effect — but critically, this is *not* evidence of no effect. The confidence interval is too wide to rule out a meaningful effect in either direction. A power analysis shows we'd need ~2,200 accounts per arm (vs ~750 available) to detect the expected 4pp lift at 80% power. **The experiment was underpowered; the honest conclusion is "inconclusive, need more data," not "the redesign failed."**

In [10]:
query = """
    SELECT *
    FROM `taskflow-growth-analytics.taskflow_dev.exp_discount_assignment`
        """
df_discount = client.query(query).to_dataframe()
df_discount.head()

,account_id,variant,chose_annual,trial_period_task_count
0,317,control,False,0
1,1913,control,False,0
2,1527,control,False,1
3,1145,control,False,1
4,794,control,False,1


In [11]:
summary_discount = df_discount.groupby("variant")["chose_annual"].agg(["count", "sum", "mean"])
summary_discount

,count,sum,mean
variant,,,
control,267,96,0.359551
treatment,230,75,0.326087


In [12]:
naive_counts = [95, 75]   # chose_annual sum for [control, treatment]
naive_totals = [267, 230]   # count for [control, treatment]

z_stat, p_value = proportions_ztest(count=naive_counts, nobs=naive_totals)

print(f"Naive z-statistic: {z_stat:.4f}")
print(f"Naive p-value: {p_value:.4f}")

Naive z-statistic: 0.6963
Naive p-value: 0.4862


In [13]:
# X = covariate, Y = outcome (as 0/1)
X = df_discount["trial_period_task_count"]
Y = df_discount["chose_annual"].astype(int)

# theta = covariance(X, Y) / variance(X)
covariance = X.cov(Y)
variance = X.var()
theta = covariance / variance

print(f"theta: {theta:.4f}")

theta: 0.0391


In [14]:
# adjusted outcome: Y_cuped = Y - theta * (X - X.mean())
X_mean = X.mean()
df_discount["y_cuped"] = Y - theta * (X - X_mean)

# compare the group means: raw vs cuped
comparison = df_discount.groupby("variant").agg(
    raw_mean=("chose_annual", "mean"),
    cuped_mean=("y_cuped", "mean")
)
comparison

,raw_mean,cuped_mean
variant,,
control,0.359551,0.352467
treatment,0.326087,0.33431


In [15]:
raw_var = df_discount["chose_annual"].astype(int).var()
cuped_var = df_discount["y_cuped"].var()

reduction = 1 - (cuped_var / raw_var)

print(f"Raw variance:   {raw_var:.4f}")
print(f"CUPED variance: {cuped_var:.4f}")
print(f"Variance reduction: {reduction:.1%}")

Raw variance:   0.2261
CUPED variance: 0.1990
Variance reduction: 12.0%


In [20]:
from scipy import stats

# split the cuped outcome by arm
control_cuped = df_discount[df_discount["variant"] == "control"]["y_cuped"]
treatment_cuped = df_discount[df_discount["variant"] == "treatment"]["y_cuped"]

t_stat, p_value_cuped = stats.ttest_ind(control_cuped, treatment_cuped)

print(f"CUPED t-statistic: {t_stat:.4f}")
print(f"CUPED p-value: {p_value_cuped:.4f}")
print(f"(compare to naive p-value: 0.4862)")

CUPED t-statistic: 0.4521
CUPED p-value: 0.6514
(compare to naive p-value: 0.4862)


In [21]:
import numpy as np

# --- raw effect: difference in means, and its standard error ---
raw_control = df_discount[df_discount["variant"] == "control"]["chose_annual"].astype(int)
raw_treatment = df_discount[df_discount["variant"] == "treatment"]["chose_annual"].astype(int)

def diff_and_se(control, treatment):
    diff = treatment.mean() - control.mean()
    se = np.sqrt(control.var(ddof=1)/len(control) + treatment.var(ddof=1)/len(treatment))
    return diff, se

raw_diff, raw_se = diff_and_se(raw_control, raw_treatment)
cuped_diff, cuped_se = diff_and_se(control_cuped, treatment_cuped)

print(f"RAW:   effect = {raw_diff:+.4f},  standard error = {raw_se:.4f}")
print(f"CUPED: effect = {cuped_diff:+.4f},  standard error = {cuped_se:.4f}")
print(f"Standard error reduced by: {1 - cuped_se/raw_se:.1%}")

RAW:   effect = -0.0335,  standard error = 0.0427
CUPED: effect = -0.0182,  standard error = 0.0400
Standard error reduced by: 6.4%


## Experiment 2: Trial Discount Framing — Results (with CUPED)

**Question:** Does "2 months free" vs "Save 20%" framing change annual-plan selection?

**Naive result:** Control 36.0% vs Treatment 32.6% annual selection. Raw effect −3.3pp, SE 0.0427, p = 0.486 — inconclusive, and the point estimate points the "wrong" way (noise-inflated).

**CUPED adjustment** (covariate: `trial_period_task_count`, correlation ~0.35 with outcome):
- Variance reduced **12.0%** (≈ correlation²)
- Standard error reduced **6.4%** (0.0427 → 0.0400)
- Effect estimate moved toward truth: −3.3pp → −1.8pp

**Conclusion:** CUPED worked as designed — it lowered the standard error and de-biased the estimate using pre-experiment data, equivalent to ~12% more sample for free. However, the experiment remained underpowered, so the result stayed inconclusive. The takeaway is methodological: CUPED improves *precision* (standard error), which is visible even when the p-value doesn't cross 0.05. Demonstrating CUPED's benefit via the standard error rather than the p-value is the statistically correct framing.

In [22]:
query = """
    SELECT *
    FROM `taskflow-growth-analytics.taskflow_dev.exp_nudge_assignment`
"""

df_nudge = client.query(query).to_dataframe()

# Rebuild the three group label
def label_group(row):
    if row["exp3_nudged"]:
        return "nudged"
    elif row["crossed_threshold"]:
        return "crossed_before_nudge"
    else:
        return "never_crossed"

df_nudge["group"] = df_nudge.apply(label_group, axis=1)
df_nudge.groupby("group")["late_upgraded"].agg(["count", "sum", "mean"])

,count,sum,mean
group,,,
crossed_before_nudge,53,31,0.584906
never_crossed,1329,82,0.061701
nudged,62,36,0.580645


In [23]:
# extract the three group rates
rate_never   = 0.061701   # never_crossed
rate_before  = 0.584906   # crossed_before_nudge (heavy users, NOT nudged)
rate_nudged  = 0.580645   # nudged (heavy users, nudged)

# NAIVE (wrong): nudged vs never_crossed -- confounded
naive_effect = rate_nudged - rate_never

# CONTROLLED (correct): nudged vs crossed_before_nudge -- confounder held constant
controlled_effect = rate_nudged - rate_before

print(f"NAIVE estimate (nudged - never_crossed):        {naive_effect:+.3f}  ({naive_effect*100:+.1f}pp)")
print(f"CONTROLLED estimate (nudged - crossed_before):  {controlled_effect:+.3f}  ({controlled_effect*100:+.1f}pp)")

NAIVE estimate (nudged - never_crossed):        +0.519  (+51.9pp)
CONTROLLED estimate (nudged - crossed_before):  -0.004  (-0.4pp)


In [24]:
# matched comparison: nudged vs crossed_before_nudge
matched = df_nudge[df_nudge["group"].isin(["nudged", "crossed_before_nudge"])]

nudged_upgrades = [36, 31]      # upgrades: [nudged, crossed_before]
nudged_totals   = [62, 53]      # totals:  [nudged, crossed_before]

z_stat, p_value = proportions_ztest(count=nudged_upgrades, nobs=nudged_totals)

print(f"Matched comparison z-statistic: {z_stat:.4f}")
print(f"Matched comparison p-value: {p_value:.4f}")

# confidence interval for the difference
ci_low, ci_upp = confint_proportions_2indep(
    count1=36, nobs1=62,   # nudged
    count2=31, nobs2=53,   # crossed_before
    method="wald"
)
print(f"95% CI (nudged - crossed_before): [{ci_low:+.4f}, {ci_upp:+.4f}]")

Matched comparison z-statistic: -0.0462
Matched comparison p-value: 0.9632
95% CI (nudged - crossed_before): [-0.1850, +0.1765]


In [25]:
import statsmodels.formula.api as smf

# restrict to the two heavy-user groups (this restriction IS the confounder control)
matched = df_nudge[df_nudge["group"].isin(["nudged", "crossed_before_nudge"])].copy()

# treatment indicator: 1 if nudged, 0 if crossed-before (not nudged)
matched["nudged_flag"] = (matched["group"] == "nudged").astype(int)
matched["upgraded_flag"] = matched["late_upgraded"].astype(int)

# logistic regression: does being nudged predict upgrading, among heavy users?
model = smf.logit("upgraded_flag ~ nudged_flag", data=matched).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.679427
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:          upgraded_flag   No. Observations:                  115
Model:                          Logit   Df Residuals:                      113
Method:                           MLE   Df Model:                            1
Date:                Thu, 13 Aug 2026   Pseudo R-squ.:               1.365e-05
Time:                        19:11:41   Log-Likelihood:                -78.134
converged:                       True   LL-Null:                       -78.135
Covariance Type:            nonrobust   LLR p-value:                    0.9632
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.3429      0.279      1.230      0.219      -0.203       0.889
nudged_flag    -0.0175    

## Experiment 3: Usage-Threshold Upgrade Nudge — Causal Inference

**Question:** Did the in-app upgrade nudge increase upgrades among heavy Free users?

**The confounding trap:** The nudge was NOT randomized — it was shown to accounts that crossed a usage threshold. Heavy users self-selected into "nudged," and heavy users upgrade more regardless.

**Naive estimate** (nudged vs never-crossed): **+51.9pp** — a fake ~9x lift, entirely driven by confounding.

**Confounder-controlled estimate** (nudged vs crossed-before-nudge — both heavy users): **−0.4pp** — essentially zero. Confirmed three independent ways:
- Raw matched rates: 58.1% vs 58.5%
- Two-proportion z-test: p = 0.963
- Logistic regression: coef = −0.0175, p = 0.963

**Conclusion:** The nudge's true effect is statistically indistinguishable from zero. The naive +52pp was almost entirely confounding — the nudge mostly reached users who were going to upgrade anyway. (Matched CI [−18.5pp, +17.6pp] is wide due to small samples, so we can rule out the 52pp claim confidently but can't pin the exact near-zero effect precisely.)